In [1]:
import json
import os
from dataclasses import asdict
from typing import Tuple, Callable

import pandas as pd

from tiu_phi_3_5_mini.data_management import load_train_validation_splits, ActivationsDataSelector, ProbeTrainScenario, \
    train_scenarios_spec_path
from tiu_phi_3_5_mini.direction_learning import ReconLosses, compute_recon_losses, DirVectors
from tiu_phi_3_5_mini.evaluation_utils import MetricsForDatasetProbes, evaluate_classifier_performance
from tiu_phi_3_5_mini.logging_setup import create_logger
from tiu_phi_3_5_mini.phi_3_5_constants import dsets_index_path, probes_folder, \
    train_split_classification_metrics_path, validation_split_classification_metrics_path, baseline_probes_folder, \
    directions_reconstruction_losses_path, analysis_results_folder
from tiu_phi_3_5_mini.phi_3_5_probe import ProbesForScenario, train_probes_for_dset, PolarityAwareTruthProbe

In [2]:
logger = create_logger(__name__)

In [3]:
dsets_index_df = pd.read_csv(dsets_index_path, index_col="Idx")
num_dsets = dsets_index_df.shape[0]
train_valid_split_specs = load_train_validation_splits()

In [4]:
data_selector = ActivationsDataSelector(dsets_index_df, train_valid_split_specs)

In [5]:
train_scenarios: list[ProbeTrainScenario] = []
if train_scenarios_spec_path.exists():
    logger.info(f"loading probe-training scenarios from file {train_scenarios_spec_path}")
    with train_scenarios_spec_path.open("r") as f:
        raw_scenarios_specs = json.load(f)
    train_scenarios = [ProbeTrainScenario(**raw_scenario_spec) for raw_scenario_spec in raw_scenarios_specs]

2025-02-18 13:55:53,452;__main__;INFO:loading probe-training scenarios from file train_scenarios_spec.json


In [6]:
if len(train_scenarios) == 0:
    logger.info("creating probe-training scenarios collection from scratch")
    min_rows_for_single_polarity_scenario = 1000
    min_rows_for_mixed_polarity_scenario = 500
    get_n_rows: Callable[[int], int] = lambda dset_idx_num: data_selector.grab_all_data_for_dset(dset_idx_num).truth_labels.shape[0]
    
    multi_topic_scenarios_folder_nm = "cross_topic"
    multi_topic_scenarios_categ_names = ["animal_class", "facts", "inventors"]
    
    for dset_idx, dset_dtls in dsets_index_df.iterrows():
        # technically, "common_claim_true_false" has on the order of ~180 records containing logical negation, but it 
        #  doesn't matter for this purpose because it'll pass this check based on record count anyway (4450 > 1000)
        if get_n_rows(dset_idx) >= min_rows_for_single_polarity_scenario:
            train_scenarios.append(ProbeTrainScenario(dset_dtls["Categ_Folder"], os.path.splitext(dset_dtls["Dataset_File"])[0], [dset_idx]))
    
    multi_topic_scenarios_affirm_dset_idxs = []
    multi_topic_scenarios_neg_dset_idxs = []
    multi_topic_scenarios_conj_dset_idxs = []
    
    for topic_nm, topic_dset_idxs in data_selector.dset_idxs_for_6way_topics.items():
        affirm_idx, neg_idx = topic_dset_idxs["affirm"], topic_dset_idxs["neg"]
        conj_idx, disj_idx = topic_dset_idxs["conj"], topic_dset_idxs["disj"]
        
        n_affirm, n_neg, n_conj, n_disj = get_n_rows(affirm_idx), get_n_rows(neg_idx), get_n_rows(conj_idx), get_n_rows(disj_idx) 
        
        if n_affirm+n_neg >= min_rows_for_mixed_polarity_scenario:
            train_scenarios.append(ProbeTrainScenario(topic_nm, "affirm_neg", [affirm_idx, neg_idx]))
        
        if n_neg+n_conj >= min_rows_for_mixed_polarity_scenario:
            train_scenarios.append(ProbeTrainScenario(topic_nm, "neg_conj", [neg_idx, conj_idx]))
        
        if n_neg+n_disj >= min_rows_for_mixed_polarity_scenario:
            train_scenarios.append(ProbeTrainScenario(topic_nm, "neg_disj", [neg_idx, disj_idx]))
        
        if n_affirm+n_neg+n_conj+n_disj >= min_rows_for_mixed_polarity_scenario:
            train_scenarios.append(ProbeTrainScenario(topic_nm, "affirm_neg_conj_disj", [affirm_idx, neg_idx, conj_idx, disj_idx]))
        
        if topic_nm in multi_topic_scenarios_categ_names:
            multi_topic_scenarios_affirm_dset_idxs.append(affirm_idx)
            multi_topic_scenarios_neg_dset_idxs.append(neg_idx)
            multi_topic_scenarios_conj_dset_idxs.append(conj_idx)
    
    train_scenarios.append(ProbeTrainScenario(
        multi_topic_scenarios_folder_nm, "multi_topic_affirm_neg", multi_topic_scenarios_affirm_dset_idxs + multi_topic_scenarios_neg_dset_idxs
    ))
    
    train_scenarios.append(ProbeTrainScenario(
        multi_topic_scenarios_folder_nm, "multi_topic_affirm_neg_conj", 
        multi_topic_scenarios_affirm_dset_idxs + multi_topic_scenarios_neg_dset_idxs + multi_topic_scenarios_conj_dset_idxs
    ))
    
    train_scenarios.append(ProbeTrainScenario(
        multi_topic_scenarios_folder_nm, "multi_topic_affirm_neg_plus_smaller_than_and_common_claim_t_f", 
        multi_topic_scenarios_affirm_dset_idxs + multi_topic_scenarios_neg_dset_idxs + 
        [data_selector.idxs_for_other_dsets["smaller_than"], data_selector.idxs_for_other_dsets["common_claim_true_false"]]
    ))
    
    with train_scenarios_spec_path.open("w") as f:
        json.dump(list(map(asdict, train_scenarios)), f, indent=2)

In [7]:
probes_folder.mkdir(exist_ok=True)
baseline_probes_folder.mkdir(exist_ok=True)
analysis_results_folder.mkdir(exist_ok=True)

In [8]:
# inner list index is split variant index (which train-validation split of the data was used to train that group of comparable probes)
probes: dict[Tuple[int,...], list[ProbesForScenario]] = { scenario.scenario_key(): [] for scenario in train_scenarios}
directions_recon_losses: dict[Tuple[int,...], list[ReconLosses]] = {scenario.scenario_key(): [] for scenario in train_scenarios}
all_dsets_train_metrics: dict[Tuple[int,...], list[MetricsForDatasetProbes]] = {scenario.scenario_key(): [] for scenario in train_scenarios}
all_dsets_val_metrics: dict[Tuple[int,...], list[MetricsForDatasetProbes]] = {scenario.scenario_key(): [] for scenario in train_scenarios}

In [9]:
def from_ttpd_probe(ttpd_probe: PolarityAwareTruthProbe) -> DirVectors:
    return DirVectors(ttpd_probe.truth_dir, ttpd_probe.polarity_dir)

In [ ]:
for scenario in train_scenarios:
    for split_variant_idx in range(data_selector.num_split_variants):
        logger.info(f"starting probe trainings for split variant index {split_variant_idx} of scenario {scenario.result_folder_name}-{scenario.scenario_name}")
        train_data, validation_data = data_selector.select_train_validation_for_scenario(scenario.src_dset_idxs, split_variant_idx)
        trained_probes = train_probes_for_dset(scenario.result_folder_name, scenario.scenario_name, split_variant_idx, train_data, validation_data)
        key = scenario.scenario_key()
        probes[key].append(trained_probes)
        
        ttpd_recon_losses = compute_recon_losses(from_ttpd_probe(trained_probes.ttpd_probe), train_data, validation_data)
        directions_recon_losses[key].append(ttpd_recon_losses)
        
        train_set_metrics = evaluate_classifier_performance(trained_probes, train_data.activations, train_data.truth_labels)
        all_dsets_train_metrics[key].append(train_set_metrics)
        val_set_metrics = evaluate_classifier_performance(trained_probes, validation_data.activations, validation_data.truth_labels)
        all_dsets_val_metrics[key].append(val_set_metrics)

2025-02-18 13:55:53,479;__main__;INFO:starting probe trainings for split variant index 0 of scenario cities-cities
2025-02-18 13:55:53,500;tiu_phi_3_5_mini.phi_3_5_probe;INFO:training the layer18 probe for 1196 records of data cities in the location trained_probes\cities
2025-02-18 13:55:53,504;tiu_phi_3_5_mini.direction_learning;INFO:doing direction-learning for 1196 records of data
2025-02-18 13:56:07,157;tiu_phi_3_5_mini.phi_3_5_probe;INFO:scaling learning rate up from 1.250000e-05 to 1.625000e-05 at epoch 1024 because loss (avg'd over 10 timesteps) has improved by inf since 1024 epochs ago and last 1024 epochs have included a minimal number of stagnant or backsliding epochs
2025-02-18 13:56:20,405;tiu_phi_3_5_mini.phi_3_5_probe;INFO:scaling learning rate up from 1.625000e-05 to 2.112500e-05 at epoch 2048 because loss (avg'd over 10 timesteps) has improved by 1.773807e-02 since 1024 epochs ago and last 1024 epochs have included a minimal number of stagnant or backsliding epochs
2025

In [ ]:
serialized_recon_losses = {
    (" ".join(map(str, scenario_id))): [asdict(recon_losses) for recon_losses in recon_losses_lst] for scenario_id, recon_losses_lst in directions_recon_losses.items()
}
with directions_reconstruction_losses_path.open("w") as f:
    json.dump(serialized_recon_losses, f, indent=2)

In [ ]:
serialized_train_metrics = { (" ".join(map(str, scenario_id))): [metrics.to_dict() for metrics in metrics_lst] for scenario_id, metrics_lst in all_dsets_train_metrics.items() }
serialized_val_metrics = { (" ".join(map(str, scenario_id))): [metrics.to_dict() for metrics in metrics_lst] for scenario_id, metrics_lst in all_dsets_val_metrics.items() }

with train_split_classification_metrics_path.open("w") as f:
    json.dump(serialized_train_metrics, f, indent=2)
with validation_split_classification_metrics_path.open("w") as f:
    json.dump(serialized_val_metrics, f, indent=2)